# 1- Extractor ONNX export

Exports **extractor-only** ONNX (no LightGlue). 
Use `uv sync --group export` (or `--all-groups`) so `torch`, `torchvision`, and `onnx` are available.

**CUDA / PyTorch:** match Jetson or desktop CUDA to the PyTorch wheel from [pytorch.org](https://pytorch.org/get-started/locally/).

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
if CWD.name == "notebooks":
    LG = CWD.parent
    REPO_ROOT = LG.parent
elif CWD.name == "LightGlue-ONNX-Jetson":
    LG = CWD
    REPO_ROOT = CWD.parent
else:
    raise SystemExit("Run Jupyter with cwd = LightGlue-ONNX-Jetson or LightGlue-ONNX-Jetson/notebooks")

WEIGHTS = REPO_ROOT / "weights"
REF = REPO_ROOT / "ref"
OUT_DIR = LG / "weights"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("LG", LG)
print("REPO_ROOT", REPO_ROOT)
print("WEIGHTS", WEIGHTS, WEIGHTS.exists())
print("REF", REF, REF.exists())
print("OUT_DIR", OUT_DIR)

### I/O contract (registry)

Each extractor has a fixed **input name** (`images`), channel count, spatial divisor for padding, minimum ONNX opset, and **output tensor names**.

In [ ]:
import sys

if str(LG) not in sys.path:
    sys.path.insert(0, str(LG))

from lightglue_dynamo.extractor_export import (
    EXTRACTOR_REGISTRY,
    build_dynamic_axes,
    export_extractor_onnx,
    list_supported_extractors,
    onnx_op_type_counts,
    validate_onnx,
)

for eid, spec in sorted(EXTRACTOR_REGISTRY.items()):
    print(
        f"{spec.id:16} in={spec.input_name} C={spec.input_channels} div={spec.spatial_divisor} "
        f"opset>={spec.min_opset} out=({', '.join(spec.output_names)})"
    )
print("Supported:", list_supported_extractors())

### Preprocessor test (numpy → NCHW float32)

Uses OpenCV BGR batch `(N,H,W,3)` consistent with `cv2.imread`.

In [ ]:
import numpy as np

try:
    import cv2
except ImportError:
    raise SystemExit("Install opencv-python")

asset = LG / "assets" / "sacre_coeur1.jpg"
if not asset.is_file():
    img = np.random.randint(0, 255, (2, 480, 768, 3), dtype=np.uint8)
else:
    im = cv2.imread(str(asset))
    im = cv2.resize(im, (768, 480))
    img = np.stack([im, im], axis=0)

for eid in ("superpoint", "superpoint_open", "disk", "aliked_n16"):
    spec = EXTRACTOR_REGISTRY[eid]
    t = spec.preprocess(img)
    print(eid, t.shape, t.dtype, float(t.min()), float(t.max()))

### Export helpers

In [ ]:
spec = EXTRACTOR_REGISTRY["superpoint_open"]
axes = build_dynamic_axes(
    input_name=spec.input_name,
    output_names=spec.output_names,
    dynamic_batch=True,
    dynamic_hw=False,
)
axes

## Example: SuperPoint-Open (multi-batch trace, B=2)

Requires `WEIGHTS / superpoint_v6_from_tf.pth`.

In [ ]:
out = OUT_DIR / "euroc" / "superpoint_open_b2_h480_w768_k256.onnx"

export_extractor_onnx(
    "superpoint_open",
    out,
    weights_root=WEIGHTS,
    batch_size=2,
    height=480,
    width=768,
    max_keypoints=256,
    opset=17,
    dynamic_batch=False,
    device="cpu",
)
validate_onnx(out, run_shape_inference=True)
print("Wrote", out)
print("Top ONNX ops:", onnx_op_type_counts(out).most_common(15))

## Example: ALIKED-n16 (opset ≥ 19, DeformConv)

Requires `WEIGHTS / aliked-n16.pth`.

In [ ]:
aliked_weights = WEIGHTS / "aliked-n16.pth"
if not aliked_weights.is_file():
    print("Skip: missing", aliked_weights)
else:
    out = OUT_DIR / "euroc" / "aliked_n16_b2_h480_w768_k256.onnx"
    export_extractor_onnx(
        "aliked_n16",
        out,
        weights_root=WEIGHTS,
        batch_size=2,
        height=480,
        width=768,
        max_keypoints=256,
        opset=19,
        device="cpu",
    )
    validate_onnx(out)
    print("DeformConv present:", onnx_op_type_counts(out).get("DeformConv", 0))

## RaCo: checkpoint schema (no descriptors)

The published `raco.pth` contains only backbone + score head weights (keypoints + scores).
A separate matcher checkpoint (`raco_aliked_lightglue.pth`) does not add descriptor weights;
deployment pairs RaCo keypoints with a separate descriptor network (e.g. ALIKED).


In [ ]:
import torch

p = WEIGHTS / "raco.pth"
if p.is_file():
    sd = torch.load(p, map_location="cpu")
    if isinstance(sd, dict):
        keys = list(sd.keys())
        print("Num tensors:", len(keys))
        print("Sample keys:", keys[:12])
        has_desc = any("desc" in k.lower() for k in keys)
        print("State dict mentions 'desc':", has_desc)
else:
    print("Missing", p)

### Example: RaCo

In [ ]:
raco_w = WEIGHTS / "raco.pth"
if raco_w.is_file():
    out = OUT_DIR / "euroc" / "raco_b2_h480_w768_k256.onnx"
    export_extractor_onnx(
        "raco", out, weights_root=WEIGHTS,
        batch_size=2, height=480, width=768, max_keypoints=256, device="cpu",
    )
    validate_onnx(out)
    print("raco", out.name, onnx_op_type_counts(out).most_common(8))
else:
    print("Skip raco: missing", raco_w)


### Example: xFeat

In [ ]:
xfeat_w = WEIGHTS / "xfeat.pt"
if xfeat_w.is_file():
    out = OUT_DIR / "euroc" / "xfeat_b2_h480_w768_k256.onnx"
    export_extractor_onnx(
        "xfeat", out, weights_root=WEIGHTS,
        batch_size=2, height=480, width=768, max_keypoints=256, device="cpu",
    )
    validate_onnx(out)
    print("xfeat", out.name, onnx_op_type_counts(out).most_common(8))
else:
    print("Skip xfeat: missing", xfeat_w)

## ORT inference comparison — superpoint_open, raco, xfeat

Runs ONNX Runtime with `["CUDAExecutionProvider", "CPUExecutionProvider"]` on both `sacre_coeur1.jpg` and `sacre_coeur2.jpg` (resized to 480×768). Keypoints are coloured by score (viridis, 0→1).


In [ ]:
import numpy as np
import onnxruntime as ort
import cv2
import matplotlib.pyplot as plt

PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]
H, W = 480, 768

asset_dir = LG / "assets"
img_files  = ["sacre_coeur1.jpg", "sacre_coeur2.jpg"]
imgs_bgr   = [cv2.resize(cv2.imread(str(asset_dir / f)), (W, H)) for f in img_files]

model_defs = [
    ("superpoint_open", OUT_DIR / "euroc" / "superpoint_open_b2_h480_w768_k256.onnx"),
    ("raco",            OUT_DIR / "euroc" / "raco_b2_h480_w768_k256.onnx"),
    ("xfeat",           OUT_DIR / "euroc" / "xfeat_b2_h480_w768_k256.onnx"),
]

fig, axes = plt.subplots(len(model_defs), len(imgs_bgr), figsize=(12, 12), dpi=96)
fig.suptitle("ORT Extractor Comparison  (H=480, W=768, batch=2)", fontsize=13)

for row_i, (eid, onnx_path) in enumerate(model_defs):
    spec         = EXTRACTOR_REGISTRY[eid]
    img_rgb_list = [img[..., ::-1] for img in imgs_bgr]

    x    = np.concatenate([spec.preprocess(img) for img in imgs_bgr], axis=0)  # (2, C, H, W)
    sess = ort.InferenceSession(str(onnx_path), providers=PROVIDERS)
    outs = sess.run(None, {spec.input_name: x})

    kpts_b, scores_b = outs[0], outs[1]       # (B,K,2)  (B,K)
    if len(outs) >= 4:                         # has num_keypoints output
        num_valid = [int(n) for n in outs[3]]
    else:                                      # raco: valid = score > 0
        num_valid = [int((s > 0).sum()) for s in scores_b]

    for col_i, img_rgb in enumerate(img_rgb_list):
        ax  = axes[row_i, col_i]
        n   = num_valid[col_i]
        ax.imshow(img_rgb)
        if n > 0:
            kpts   = kpts_b[col_i, :n]
            scores = scores_b[col_i, :n]
            sc = ax.scatter(kpts[:, 0], kpts[:, 1], c=scores,
                            cmap="plasma", s=4, linewidths=2, vmin=2, vmax=3)
            plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
        ax.set_title(f"{eid}  |  {img_files[col_i]}  ({n} kpts)", fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()
